# *1. Data Analysis in HEP II - Introduction*

![image.gif](assets/HiggsGammaGamma.gif)


This will be the introductory session for the Data Analysis II seminar, being more of a tutorial for setting up all of the tools we will need in the future labs and analyses. 


## **The standard software used in research**

There are a lot of frameworks and programs used in HEP, with a lot of collaborations opting for personalized workflows, so there is no wrong option if your tools get you from point A to point B. 

There are some that are widely regarded as the golden standard in HEP and we will employ some of these in the scopes of this seminar.

1. ***ROOT framework***

[ROOT: Analyzing petabytes of data, scientifically.](https://root.cern/)

This is the poster child when it comes to HEP software. It was developed with the purpose of handling very large volumes of data and now sits at the basis of most of the tools we use, from plotting and fitting data, to performing complex statistical analysis (we will see later in the semester).


2. ***Python***

[Python](https://www.python.org/)


Modern, with great library support and very easy to use, a lot of people migrated to Python. We will mostly be using this programming language in our seminars, with some exeptions in the latter parts.



## **Setting up the environment for work**

In order to keep modular and scalable working environments, we will be using virtual Python environments, built using [Conda](https://docs.conda.io/projects/conda/en/stable/index.html). Depending on the OS you are working on, take the corresponding installation path. On Linux and MacOS, everything works out of the box. On Windows we will need to use [Windows Subsystem for Linux](https://learn.microsoft.com/ro-ro/windows/wsl/) (WSL) in order to get the maximum out of our tools.

### *Linux*

Download the shell script and install it with:
```bash
    wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    bash Miniconda3-latest-Linux-x86_64.sh
```

After that, restart the terminal using:
```bash
    source ~/.bashrc
```

Create the environment with all of the dependencies in the [`hep_env.yml`](hep_env.yml) file. You will need around 6 GB of free storage space:
```bash
    conda create -f hep_env.yml
```

After it's done installing everything, you can activate the environment by calling:
```bash
    conda activate hep_stats
```



### *MacOS*

Download the shell package (***only for Apple Silicon CPUs! i.e. M1, M2...***) and install it with with:
```bash
    wget https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
    bash Miniconda3-latest-MacOSX-arm64.sh
```

***If your Macbook has an Intel CPU, use this one instead!***
```bash
    wget https://repo.anaconda.com/miniconda/Miniconda3-latest-MacOSX-arm64.sh
    back Miniconda3-latest-MacOSX-arm64.sh
```

After that, restart the terminal using:
```bash
    source ~/.zshrc
```

Create the environment with all of the dependencies in the [`hep_env.yml`](hep_env.yml) file. You will need around 6 GB of free storage space:
```bash
    conda create -f hep_env.yml
```

After it's done installing everything, you can activate the environment by calling:
```bash
    conda activate hep_stats
```


### *Windows*

In order to install the environment and the tools we'll be working with, we will need to install WSL to work in a Linux environment which will make our lives much easier. We do this by opening PowerShell and running:
```bash
    wsl --install
```
This will install [Ubuntu](https://ubuntu.com/download/desktop) by default. If you prefer other linux distributions, you can check the available ones by running:
```bash
    wsl --list --online
```
After that, you need to reboot your Windows system and start the virtual machine by simply running in PowerShell:
```bash
    wsl
```
You will be asked to configure your user, after which you can follow the ***exact steps as in the Linux installation presented [above](###-*linux*)***.

If you're using WSL, you might also need to install a C++ compiler. You can check if you already have it installed by running:
```bash
    gcc --version
```

If you don't have it on your VM, install it using:
```bash
    sudo apt install gcc
```

A friendly advice for working with WSL: use VS Code and configure a WSL target in the remote explorer, for much better quality of life when working with the IDE.

If there's any trouble, feel free to ask for help!



## **Small introduction to how plots and fits are made**

Now that we have all the tools at our disposal, we will go over a toy example. We will not use real data yet, we will generate some data starting from underlying distributions and try to see how to extract quantities from our plots. Just try to follow what the program is doing, we will go over the theoretical aspects in the later seminars.

First of all, we will import the libraries we will need (we will add and subtract to them in the future, depending on what our goals are):
 - `matplotlib` is our go-to in Python for making plots
 - `mplhep` is an add-on for matplotlib that stylizes plots similarly to the large HEP experiments.
 - `numpy` is used for all things mathematics
 - `scipy` will be used for fitting and more complex functions

In [ ]:
import matplotlib.pyplot as plt
import mplhep as mh
import numpy as np
import scipy

Now we need to generate data. We will try to replicate the image you have seen above with the invariant mass of the Higgs boson. The background shape will be an exponential, with a Gaussian signal bump where we expect the Higgs to be.

In [ ]:
# We generate some toy data for us to play with. The background is an exponential distribution which we shifted with 100 GeV to the right
bkg_events = np.random.exponential(30, 10000) + 100
# The signal is a Gaussian distribution centered at 125 GeV with a width of 2 GeV
sig_events = np.random.normal(125, 2, 400)
# We put the data together and only keep the events that are in the range [100, 160] GeV
data = np.concatenate([bkg_events, sig_events])
data = data[(data > 100) & (data < 160)]

# Here we bin the data points into a histogram
counts, bin_edges = np.histogram(data, bins=60, range=(100, 160))
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

In [ ]:
# Here we create our figure and do some magic with mplhep, which stylizes our plots to look like the ones in literature without that much of a hassle
mh.set_style("ATLAS")
fig, ax = plt.subplots()
mh.histplot(counts, bin_edges, yerr=True, histtype='errorbar', ax=ax, color='black')
mh.atlas.label('Toy data for education', data=True, lumi=150, com=7, ax=ax, loc=0)

# We make an educated guess for the initial values of the paramters of our distribution and then we perform the fit on the data
initial_params = [100, 125, 2, 1000, 30]
popt, pcov = scipy.optimize.curve_fit(exp_gaus, bin_centres, counts, p0=initial_params)
perr = np.sqrt(np.diag(pcov))

# After that, we make a high-resolution array such that our fitted functions look nice on the plot
x = np.linspace(100, 160, 1000)
ax.plot(x, exp_gaus(x, 0, 0, 1, popt[3], popt[4]), label='Background', color='blue', linestyle='--')
ax.plot(x, exp_gaus(x, *popt), label='Fit', color='red')
ax.set_xlabel(r"Invariant mass $m_{\gamma\gamma}$ [GeV]")
ax.set_ylabel(f"Events / {bin_edges[1]-bin_edges[0]} GeV")
mh.mpl_magic()
plt.show()

Congrats! You just made a HEP-style plot! (even though we used fabricated data) The next logical step is finding out how to read actual files, and visualize the observables in processes.

We will do this using tools like `awkward`, `uproot` and `atlasopenmagic`. [ATLAS Open Data](https://opendata.atlas.cern) is a great platform full of strong educational material. It might be worth to take a gander over there, to look up documentation or other stuff.

Let's choose a simple process we will analyse over the following 2-3 weeks, $Z → μ^+μ^-$. We will start by initialising `atlasopenmagic` and all of its dependencies (this is a one-time thing).

In [ ]:
import atlasopenmagic as atom
atom.install_from_environment()
import uproot
import awkward as ak

Now that we have everything in place, we can look what data releases we have available to work with. We will choose the 2025 Educational release for now.

In [ ]:
atom.available_releases()
atom.set_release('2025e-13tev-beta')

Each process has its own [metadata](https://opendata.atlas.cern/docs/data/for_research/metadata#api-for-metadata-access) that contains all of the information to help you choose what you need. For example take a look at this process and try to pick up on what seems familiar.

In [ ]:
atom.get_metadata(700323)

Now that we have the metadata, we can build the dataset specifying that we want a skimmed set that contains only 2 muons and we want to stream it from ATLAS Open Data. 

#### *Small warning!*
Caching this dataset locally will occupy some disk space. Normally these are temporary files that will be deleted by the OS after 1 week, but if you run out of space you can manually delete them without any issue after you're done working with them. Depending on what you're working on, these are stored differently. On Linux/MacOS it should somewhere in `/var` or `/tmp` and on Windows somewhere in `AppData\Local\Temp`. If it becomes a problem we can check together or you can try asking your favourite LLM for a code snippet to check where this cache is saved (might be helpful to know that it's cached using `fsspec`).

In [ ]:
defs = {
    'Data': {'dids': ['data']},
    'mu_mu': {'dids': [700323]}
}
skim = "noskim"

samples = atom.build_dataset(defs, skim=skim, protocol='https', cache=True)

Now that we loaded the dataset, we can look how many entries and variables it has.

In [ ]:
mu_mu_file = samples['mu_mu']['list'][0]

tree = uproot.open(mu_mu_file + ":analysis")

print("\n The number of entries in the tree are:", tree.num_entries)

print("\n The number of variables in the tree is:", len(tree.keys()))

Those are quite a few. We won't be needing this many for now, we'll focus on the $p_T$, $η$ and $ϕ$ of our muons.

In [ ]:
muon_variables = ['truth_muon_pt', 'truth_muon_eta', 'truth_muon_phi']

for array in tree.iterate(muon_variables, library='ak'):
    muon_pt = array["truth_muon_pt"]
    muon_eta = array["truth_muon_eta"]
    muon_phi = array["truth_muon_phi"]

Initially, the arrays are jagged, with some events having more entries than others.

In [ ]:
print(muon_pt[:5])

We flatten them to make them easier to work with.

In [ ]:
muon_pt_flat = ak.flatten(muon_pt)
muon_eta_flat = ak.flatten(muon_eta)
muon_phi_flat = ak.flatten(muon_phi)

In [ ]:
print(muon_pt_flat[:10])

Now let's make some histograms to see the distribution of the observables.

In [ ]:
fig1, ax1 = plt.subplots()
fig2, ax2 = plt.subplots()
fig3, ax3 = plt.subplots()

mu_pt = ak.to_numpy(muon_pt_flat)
bins_pt = np.linspace(0, 300, 51)
bins_pt_centers = (bins_pt[1:] + bins_pt[:-1]) / 2
ax1.hist(mu_pt, bins=bins_pt, histtype='step')
ax1.set_xlabel(r'$p_T$ [GeV]')
ax1.set_ylabel(f'Events / {bins_pt[1]-bins_pt[0]} GeV')

mu_eta = ak.to_numpy(muon_eta_flat)
bins_eta = np.linspace(-5, 5, 51)
bins_eta_centers = (bins_eta[1:] + bins_eta[:-1]) / 2
ax2.hist(mu_eta, bins=bins_eta, histtype='step')
ax2.set_xlabel(r'$\eta$')
ax2.set_ylabel(r'Events')

mu_phi = ak.to_numpy(muon_phi_flat)
bins_phi = np.linspace(-4, 4, 51)
bins_phi_centers = (bins_phi[1:] + bins_phi[:-1]) / 2
ax3.hist(mu_phi, bins=bins_phi, histtype='step')
ax3.set_xlabel(r'$\phi$ [rad]')
ax3.set_ylabel(r'Events')

plt.tight_layout()
plt.show()